
# Career–City Fit MVP: 2024 BLS OEWS + ACS Data Pipeline

This notebook does five things:

1. downloads the **2024 BLS OEWS metro/nonmetro file** and the **2024 BLS MSA definitions** helper file,
2. downloads **2024 ACS 1-year** metro-level housing/context data,
3. converts the raw inputs to CSV,
4. cleans the data and attempts to merge it into an **occupation × metro** dataset,
5. runs a first round of exploratory data analysis and produces a few charts.

## Scope

This is a **2024-only MVP**. It is designed to get one clean cross-section working end-to-end before backfilling additional years.

## Notes

- The ACS API request in this notebook works without a key for small requests, but you can optionally set `CENSUS_API_KEY`.
- The BLS OEWS metro/nonmetro file is currently linked as a ZIP archive from the BLS OEWS tables page, so this notebook downloads the ZIP, inspects its contents, and extracts the spreadsheet automatically.
- The merge uses the **BLS 2024 MSA definitions workbook** as a helper crosswalk and falls back to **normalized metro-name matching** if a direct code-based join is not available.



## 0. Optional package install

Uncomment and run this cell once if your environment is missing any of these packages.


In [1]:

%pip install -q polars pandas openpyxl requests matplotlib pyarrow


Note: you may need to restart the kernel to use updated packages.


## 1. Imports and notebook configuration

In [14]:

from __future__ import annotations

import io
import json
import math
import os
import re
import zipfile
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import requests
from IPython.display import display

pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(80)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
FIG_DIR = PROJECT_ROOT / "figures"

for path in [RAW_DIR / "bls", RAW_DIR / "acs", INTERIM_DIR, PROCESSED_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

USER_AGENT = "career-city-fit-notebook/1.0 (education project)"
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": USER_AGENT})


## 2. Configuration

In [15]:

BLS_OEWS_ZIP_URL = "https://www.bls.gov/oes/special-requests/oesm24ma.zip"
BLS_MSA_DEFINITIONS_URL = "https://www.bls.gov/oes/area_definitions_m2024.xlsx"
ACS_BASE_URL = "https://api.census.gov/data/2024/acs/acs1"

CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

BLS_ZIP_PATH = RAW_DIR / "bls" / "oesm24ma.zip"
BLS_EXTRACT_DIR = RAW_DIR / "bls" / "oesm24ma_extracted"
BLS_MSA_DEFINITIONS_PATH = RAW_DIR / "bls" / "area_definitions_m2024.xlsx"

ACS_RAW_JSON_PATH = RAW_DIR / "acs" / "acs_2024_metro_raw.json"
ACS_RAW_CSV_PATH = RAW_DIR / "acs" / "acs_2024_metro_raw.csv"

BLS_RAW_CSV_PATH = RAW_DIR / "bls" / "oews_2024_raw.csv"
BLS_CROSSWALK_CSV_PATH = INTERIM_DIR / "bls_metro_crosswalk_2024.csv"
ACS_CLEAN_CSV_PATH = INTERIM_DIR / "acs_2024_metro_context.csv"
BLS_CLEAN_CSV_PATH = INTERIM_DIR / "oews_2024_occ_metro.csv"
MERGED_CSV_PATH = PROCESSED_DIR / "occ_metro_2024_merged.csv"
UNMATCHED_BLS_METROS_CSV_PATH = INTERIM_DIR / "bls_unmatched_metros.csv"


## 3. Helper functions

In [16]:

def download_file(url: str, destination: Path, overwrite: bool = False, chunk_size: int = 1024 * 1024) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not overwrite:
        print(f"Using existing file: {destination}")
        return destination

    print(f"Downloading: {url}")
    with SESSION.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(destination, "wb") as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)

    print(f"Saved: {destination} ({destination.stat().st_size / 1_000_000:.2f} MB)")
    return destination


def sanitize_column_name(name: str) -> str:
    name = str(name).strip().lower()
    name = re.sub(r"[%/]", " ", name)
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def sanitize_columns(columns: Iterable[str]) -> list[str]:
    return [sanitize_column_name(c) for c in columns]


def normalize_metro_name(value: str | None) -> str | None:
    if value is None:
        return None
    text = str(value).strip().lower()

    replacements = {
        "&": " and ",
        "/": " ",
        ".": "",
        ", usa": "",
        " metro area": "",
        " micro area": "",
        " metropolitan statistical area": "",
        " micropolitan statistical area": "",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)

    # Remove generic words that often differ across sources.
    text = re.sub(r"\b(area|division|the)\b", " ", text)

    # Normalize whitespace / punctuation.
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def pick_first_existing(columns: list[str], candidates: list[str]) -> str | None:
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def to_numeric_expr(column_name: str, dtype: pl.DataType = pl.Float64) -> pl.Expr:
    return (
        pl.col(column_name)
        .cast(pl.Utf8)
        .str.replace_all(",", "")
        .str.replace_all(r"[^0-9.\-]", "")
        .replace("", None)
        .cast(dtype, strict=False)
        .alias(column_name)
    )


def display_pl(df: pl.DataFrame, rows: int = 8):
    display(df.head(rows).to_pandas())


def print_shape(label: str, df: pl.DataFrame):
    print(f"{label}: {df.height:,} rows × {df.width:,} cols")


def acs_params() -> dict:
    params = {
        "get": "NAME,B25064_001E,B19013_001E,B01003_001E",
        "for": "metropolitan statistical area/micropolitan statistical area:*",
    }
    if CENSUS_API_KEY:
        params["key"] = CENSUS_API_KEY
    return params


## 4. Download raw source files

In [17]:

# 4a. Download the BLS OEWS ZIP and the BLS 2024 MSA definitions workbook.
download_file(BLS_OEWS_ZIP_URL, BLS_ZIP_PATH)
download_file(BLS_MSA_DEFINITIONS_URL, BLS_MSA_DEFINITIONS_PATH)

# 4b. Download ACS 2024 1-year metro/micro data from the Census API.
print("Requesting ACS 2024 API data...")
acs_response = SESSION.get(ACS_BASE_URL, params=acs_params(), timeout=120)
acs_response.raise_for_status()

acs_json = acs_response.json()
with open(ACS_RAW_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(acs_json, f)

acs_header = acs_json[0]
acs_rows = acs_json[1:]
acs_raw_pd = pd.DataFrame(acs_rows, columns=acs_header)
acs_raw_pd.to_csv(ACS_RAW_CSV_PATH, index=False)

print(f"Saved ACS raw JSON: {ACS_RAW_JSON_PATH}")
print(f"Saved ACS raw CSV:  {ACS_RAW_CSV_PATH}")
display(acs_raw_pd.head())


Using existing file: /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/raw/bls/oesm24ma.zip
Using existing file: /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/raw/bls/area_definitions_m2024.xlsx
Requesting ACS 2024 API data...
Saved ACS raw JSON: /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/raw/acs/acs_2024_metro_raw.json
Saved ACS raw CSV:  /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/raw/acs/acs_2024_metro_raw.csv


,NAME,B25064_001E,B19013_001E,B01003_001E,metropolitan statistical area/micropolitan statistical area
0,"Aberdeen, WA Micro Area",1127,59619,77893,10140
1,"Abilene, TX Metro Area",1173,63390,181969,10180
2,"Adrian, MI Micro Area",962,71632,97746,10300
3,"Aguadilla, PR Metro Area",572,24351,250969,10380
4,"Akron, OH Metro Area",1059,71364,702209,10420


## 5. Inspect and extract the BLS OEWS ZIP

In [ ]:

BLS_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(BLS_ZIP_PATH, "r") as zf:
    names = zf.namelist()
    print("ZIP contents:")
    for name in names:
        print(" -", name)

    candidate_files = [
        name for name in names
        if name.lower().endswith((".xlsx", ".xls", ".csv", ".txt"))
    ]
    if not candidate_files:
        raise FileNotFoundError("No spreadsheet or text file was found inside the BLS ZIP.")

    # Prefer Excel if available, then CSV, then TXT.
    preferred_order = [".xlsx", ".xls", ".csv", ".txt"]
    extracted_member = None
    for ext in preferred_order:
        matches = [name for name in candidate_files if name.lower().endswith(ext)]
        if matches:
            extracted_member = matches[2]
            break

    assert extracted_member is not None
    print(f"Chosen BLS data file inside ZIP: {extracted_member}")
    zf.extract(extracted_member, path=BLS_EXTRACT_DIR)

BLS_EXTRACTED_PATH = BLS_EXTRACT_DIR / extracted_member
print("Extracted path:", BLS_EXTRACTED_PATH)


ZIP contents:
 - oesm24ma/
 - oesm24ma/BOS_M2024_dl.xlsx
 - oesm24ma/file_descriptions.xlsx
 - oesm24ma/MSA_M2024_dl.xlsx
Chosen BLS data file inside ZIP: oesm24ma/BOS_M2024_dl.xlsx
Extracted path: /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/raw/bls/oesm24ma_extracted/oesm24ma/BOS_M2024_dl.xlsx


## 6. Read the raw BLS file and convert it to CSV

In [19]:

suffix = BLS_EXTRACTED_PATH.suffix.lower()

if suffix in {".xlsx", ".xls"}:
    # Read only the first sheet. If the workbook has multiple sheets,
    # inspect them manually and switch the sheet_name if needed.
    bls_raw_pd = pd.read_excel(BLS_EXTRACTED_PATH, sheet_name=0)
elif suffix == ".csv":
    bls_raw_pd = pd.read_csv(BLS_EXTRACTED_PATH)
elif suffix == ".txt":
    # Fall back to pandas' default delimiter detection.
    bls_raw_pd = pd.read_csv(BLS_EXTRACTED_PATH, sep=None, engine="python")
else:
    raise ValueError(f"Unsupported extracted BLS file type: {suffix}")

bls_raw_pd.to_csv(BLS_RAW_CSV_PATH, index=False)

print("Raw BLS columns:")
print(list(bls_raw_pd.columns))
print(f"Raw BLS shape: {bls_raw_pd.shape[0]:,} rows × {bls_raw_pd.shape[1]:,} cols")
display(bls_raw_pd.head())
print(f"Saved raw BLS CSV: {BLS_RAW_CSV_PATH}")


Raw BLS columns:
['AREA', 'AREA_TITLE', 'AREA_TYPE', 'PRIM_STATE', 'NAICS', 'NAICS_TITLE', 'I_GROUP', 'OWN_CODE', 'OCC_CODE', 'OCC_TITLE', 'O_GROUP', 'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT', 'H_MEAN', 'A_MEAN', 'MEAN_PRSE', 'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90', 'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90', 'ANNUAL', 'HOURLY']
Raw BLS shape: 48,828 rows × 32 cols


,AREA,AREA_TITLE,AREA_TYPE,PRIM_STATE,NAICS,NAICS_TITLE,I_GROUP,OWN_CODE,OCC_CODE,OCC_TITLE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,00-0000,All Occupations,...,18.94,25,35.51,22380,29690,39400,52000,73870,NaN,NaN
1,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-0000,Management Occupations,...,40.54,58.5,80.72,44770,60730,84320,121680,167910,NaN,NaN
2,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1011,Chief Executives,...,71.58,85.16,#,114610,121930,148890,177130,#,NaN,NaN
3,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1021,General and Operations Managers,...,46.63,73.99,101.78,42680,63380,96980,153900,211700,NaN,NaN
4,100001,Northwest Alabama nonmetropolitan area,6,AL,0,Cross-industry,cross-industry,1235,11-1031,Legislators,...,*,*,*,18210,22690,51460,52020,52020,True,NaN


Saved raw BLS CSV: /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/raw/bls/oews_2024_raw.csv


## 7. Build a BLS metro crosswalk from the 2024 MSA definitions workbook

In [22]:

msa_defs_pd = pd.read_excel(BLS_MSA_DEFINITIONS_PATH)
msa_defs_pd.columns = [str(c).strip() for c in msa_defs_pd.columns]
display(msa_defs_pd.head())

msa_defs = pl.from_pandas(msa_defs_pd)
msa_defs.columns = sanitize_columns(msa_defs.columns)

# The original sheet uses a header with a trailing space: "May 2024 MSA code ".
# After sanitizing, it becomes "may_2024_msa_code".
crosswalk = (
    msa_defs
    .rename({
        "may_2024_msa_code": "cbsa_code",
        "may_2024_msa_name": "bls_msa_name",
        "state_abbreviation": "state_abbr",
        "county_name": "county_name",
    })
    .select(["cbsa_code", "bls_msa_name"])
    .drop_nulls()
    .unique()
    .with_columns([
        pl.col("cbsa_code").cast(pl.Int64, strict=False),
        pl.col("bls_msa_name").cast(pl.Utf8),
    ])
    # Keep metro codes only. BLS nonmetro helper codes are 6 digits or larger,
    # while standard CBSA metro codes are 5 digits.
    .filter(pl.col("cbsa_code") < 100000)
    .filter(~pl.col("bls_msa_name").str.to_lowercase().str.contains("nonmetropolitan"))
    .with_columns([
        pl.col("cbsa_code").cast(pl.Utf8).str.zfill(5).alias("cbsa_code"),
        pl.col("bls_msa_name").map_elements(normalize_metro_name, return_dtype=pl.Utf8).alias("metro_name_norm"),
    ])
    .sort("cbsa_code")
)

crosswalk.write_csv(BLS_CROSSWALK_CSV_PATH)

print_shape("BLS metro crosswalk", crosswalk)
display_pl(crosswalk)
print(f"Saved: {BLS_CROSSWALK_CSV_PATH}")


,FIPS code,State,State abbreviation,May 2024 MSA code,May 2024 MSA name,County code,County name
0,1,Alabama,AL,33860,"Montgomery, AL",1,Autauga County
1,1,Alabama,AL,19300,"Daphne-Fairhope-Foley, AL",3,Baldwin County
2,1,Alabama,AL,100004,Southeast Alabama nonmetropolitan area,5,Barbour County
3,1,Alabama,AL,13820,"Birmingham, AL",7,Bibb County
4,1,Alabama,AL,13820,"Birmingham, AL",9,Blount County


BLS metro crosswalk: 393 rows × 3 cols


,cbsa_code,bls_msa_name,metro_name_norm
0,10180,"Abilene, TX",abilene tx
1,10380,"Aguadilla, PR",aguadilla pr
2,10420,"Akron, OH",akron oh
3,10500,"Albany, GA",albany ga
4,10540,"Albany, OR",albany or
5,10580,"Albany-Schenectady-Troy, NY",albany schenectady troy ny
6,10740,"Albuquerque, NM",albuquerque nm
7,10780,"Alexandria, LA",alexandria la


Saved: /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/interim/bls_metro_crosswalk_2024.csv


## 8. Clean ACS into a metro-level context table

In [23]:

acs_raw = pl.read_csv(ACS_RAW_CSV_PATH)
acs_raw.columns = sanitize_columns(acs_raw.columns)

# Identify the long geography-code column produced by the ACS API.
geo_code_col = pick_first_existing(
    acs_raw.columns,
    [
        "metropolitan_statistical_area_micropolitan_statistical_area",
        "metropolitan_statistical_area",
        "micropolitan_statistical_area",
    ],
)

if geo_code_col is None:
    raise KeyError(
        "Could not identify the ACS geography code column. "
        f"Available columns: {acs_raw.columns}"
    )

acs_clean = (
    acs_raw
    .rename({
        "name": "acs_area_name",
        geo_code_col: "cbsa_code",
        "b25064_001e": "median_gross_rent",
        "b19013_001e": "median_household_income",
        "b01003_001e": "population",
    })
    .with_columns([
        pl.col("cbsa_code").cast(pl.Utf8).str.zfill(5).alias("cbsa_code"),
        to_numeric_expr("median_gross_rent", pl.Int64),
        to_numeric_expr("median_household_income", pl.Int64),
        to_numeric_expr("population", pl.Int64),
    ])
    .join(
        crosswalk.select(["cbsa_code", "bls_msa_name", "metro_name_norm"]),
        on="cbsa_code",
        how="inner",
    )
    .with_columns([
        pl.lit(2024).alias("year"),
        (pl.col("median_gross_rent") * 12).alias("annual_rent"),
        pl.col("acs_area_name").map_elements(normalize_metro_name, return_dtype=pl.Utf8).alias("acs_name_norm"),
    ])
    .select([
        "year",
        "cbsa_code",
        "bls_msa_name",
        "acs_area_name",
        "median_gross_rent",
        "annual_rent",
        "median_household_income",
        "population",
        "metro_name_norm",
        "acs_name_norm",
    ])
    .sort("cbsa_code")
)

acs_clean.write_csv(ACS_CLEAN_CSV_PATH)

print_shape("ACS clean metro context", acs_clean)
display_pl(acs_clean)
print(f"Saved: {ACS_CLEAN_CSV_PATH}")


ACS clean metro context: 393 rows × 10 cols


,year,cbsa_code,bls_msa_name,acs_area_name,median_gross_rent,annual_rent,median_household_income,population,metro_name_norm,acs_name_norm
0,2024,10180,"Abilene, TX","Abilene, TX Metro Area",1173,14076,63390,181969,abilene tx,abilene tx
1,2024,10380,"Aguadilla, PR","Aguadilla, PR Metro Area",572,6864,24351,250969,aguadilla pr,aguadilla pr
2,2024,10420,"Akron, OH","Akron, OH Metro Area",1059,12708,71364,702209,akron oh,akron oh
3,2024,10500,"Albany, GA","Albany, GA Metro Area",975,11700,56851,146051,albany ga,albany ga
4,2024,10540,"Albany, OR","Albany, OR Metro Area",1363,16356,80422,132474,albany or,albany or
5,2024,10580,"Albany-Schenectady-Troy, NY","Albany-Schenectady-Troy, NY Metro Area",1341,16092,86637,913485,albany schenectady troy ny,albany schenectady troy ny
6,2024,10740,"Albuquerque, NM","Albuquerque, NM Metro Area",1220,14640,76097,929919,albuquerque nm,albuquerque nm
7,2024,10780,"Alexandria, LA","Alexandria, LA Metro Area",899,10788,55909,148008,alexandria la,alexandria la


Saved: /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/interim/acs_2024_metro_context.csv


## 9. Clean raw BLS OEWS data into occupation × metro rows

In [24]:
bls_raw = pl.read_csv(BLS_RAW_CSV_PATH, infer_schema_length=5000, ignore_errors=True)
bls_raw.columns = sanitize_columns(bls_raw.columns)

print("Sanitized raw BLS columns:")
print(bls_raw.columns)

expected_bls_cols = [
    "area",
    "area_title",
    "area_type",
    "prim_state",
    "naics",
    "naics_title",
    "i_group",
    "own_code",
    "occ_code",
    "occ_title",
    "o_group",
    "tot_emp",
    "emp_prse",
    "jobs_1000",
    "loc_quotient",
    "pct_total",
    "pct_rpt",
    "h_mean",
    "a_mean",
    "mean_prse",
    "h_pct10",
    "h_pct25",
    "h_median",
    "h_pct75",
    "h_pct90",
    "a_pct10",
    "a_pct25",
    "a_median",
    "a_pct75",
    "a_pct90",
    "annual",
    "hourly",
]

missing_cols = [c for c in ["area", "area_title", "occ_code", "occ_title", "o_group"] if c not in bls_raw.columns]
if missing_cols:
    raise KeyError(f"Missing required BLS columns: {missing_cols}")

# Optional diagnostics: inspect the slice values before filtering
for col in ["area_type", "i_group", "own_code", "naics", "naics_title", "o_group"]:
    if col in bls_raw.columns:
        print(f"\nUnique sample values for {col}:")
        print(
            bls_raw.select(col)
            .drop_nulls()
            .unique()
            .sort(col)
            .head(20)
            .to_pandas()
        )

bls_clean = (
    bls_raw
    .select([c for c in expected_bls_cols if c in bls_raw.columns])
    .with_columns([
        pl.lit(2024).alias("year"),
        pl.col("area").cast(pl.Utf8).str.strip_chars().alias("area"),
        pl.col("area_title").cast(pl.Utf8).str.strip_chars().alias("area_title"),
        pl.col("area_type").cast(pl.Utf8).str.strip_chars().alias("area_type"),
        pl.col("prim_state").cast(pl.Utf8).str.strip_chars().alias("prim_state"),
        pl.col("naics").cast(pl.Utf8).str.strip_chars().alias("naics"),
        pl.col("naics_title").cast(pl.Utf8).str.strip_chars().alias("naics_title"),
        pl.col("i_group").cast(pl.Utf8).str.strip_chars().str.to_lowercase().alias("i_group"),
        pl.col("own_code").cast(pl.Utf8).str.strip_chars().alias("own_code"),
        pl.col("occ_code").cast(pl.Utf8).str.strip_chars().alias("occ_code"),
        pl.col("occ_title").cast(pl.Utf8).str.strip_chars().alias("occ_title"),
        pl.col("o_group").cast(pl.Utf8).str.strip_chars().str.to_lowercase().alias("o_group"),
    ])
    .with_columns([
        to_numeric_expr("tot_emp", pl.Float64),
        to_numeric_expr("emp_prse", pl.Float64),
        to_numeric_expr("jobs_1000", pl.Float64),
        to_numeric_expr("loc_quotient", pl.Float64),
        to_numeric_expr("pct_total", pl.Float64),
        to_numeric_expr("pct_rpt", pl.Float64),
        to_numeric_expr("h_mean", pl.Float64),
        to_numeric_expr("a_mean", pl.Float64),
        to_numeric_expr("mean_prse", pl.Float64),
        to_numeric_expr("h_pct10", pl.Float64),
        to_numeric_expr("h_pct25", pl.Float64),
        to_numeric_expr("h_median", pl.Float64),
        to_numeric_expr("h_pct75", pl.Float64),
        to_numeric_expr("h_pct90", pl.Float64),
        to_numeric_expr("a_pct10", pl.Float64),
        to_numeric_expr("a_pct25", pl.Float64),
        to_numeric_expr("a_median", pl.Float64),
        to_numeric_expr("a_pct75", pl.Float64),
        to_numeric_expr("a_pct90", pl.Float64),
    ])
    .filter(
        pl.col("area_title").is_not_null() &
        pl.col("occ_code").is_not_null() &
        pl.col("occ_title").is_not_null()
    )
)

# Keep only detailed occupations
bls_clean = bls_clean.filter(
    pl.col("o_group").str.contains("detail") |
    (
        pl.col("occ_code").str.contains(r"^\d{2}-\d{4}$") &
        ~pl.col("occ_code").str.ends_with("0000")
    )
)

# Keep only metro rows
# This is the main fix for your unmatched file issue.
# bls_clean = bls_clean.filter(
#     ~pl.col("area_title").str.to_lowercase().str.contains("nonmetropolitan")
# )

# Keep only cross-industry / all-industry rows when identifiable.
# This avoids retaining sector-specific slices.
cross_industry_filter = (
    pl.col("i_group").str.contains("cross") |
    pl.col("naics_title").str.to_lowercase().str.contains("cross") |
    (pl.col("naics") == "000000")
)

cross_industry_count = bls_clean.filter(cross_industry_filter).height
if cross_industry_count > 0:
    bls_clean = bls_clean.filter(cross_industry_filter)
    print(f"Kept cross-industry rows only: {cross_industry_count:,} rows")
else:
    print("No obvious cross-industry rows found from i_group/naics fields; keeping current rows for now.")

# Create normalized metro key and a CBSA guess from AREA
bls_clean = bls_clean.with_columns([
    pl.col("area").str.extract(r"(\d{5})$", 1).alias("cbsa_code_guess"),
    pl.col("area_title").map_elements(normalize_metro_name, return_dtype=pl.Utf8).alias("metro_name_norm"),
])

# Attempt 1: code-based join
bls_joined_by_code = (
    bls_clean
    .join(
        crosswalk.select(["cbsa_code", "bls_msa_name"]),
        left_on="cbsa_code_guess",
        right_on="cbsa_code",
        how="left",
    )
)

code_join_ratio = bls_joined_by_code.select(
    pl.col("bls_msa_name").is_not_null().mean()
).item()
print(f"Code-based join coverage: {code_join_ratio:.2%}")

# Attempt 2: normalized metro-name join
bls_joined_by_name = (
    bls_clean
    .join(
        crosswalk.select(["cbsa_code", "bls_msa_name", "metro_name_norm"]),
        on="metro_name_norm",
        how="left",
    )
)

name_join_ratio = bls_joined_by_name.select(
    pl.col("cbsa_code").is_not_null().mean()
).item()
print(f"Name-based join coverage: {name_join_ratio:.2%}")

if code_join_ratio >= max(0.85, name_join_ratio):
    print("Using code-based metro join.")
    bls_joined = (
        bls_joined_by_code
        .rename({"cbsa_code_guess": "cbsa_code"})
        .drop("bls_msa_name")
        .join(
            crosswalk.select(["cbsa_code", "bls_msa_name"]),
            on="cbsa_code",
            how="left",
        )
    )
else:
    print("Using normalized metro-name join.")
    bls_joined = bls_joined_by_name

bls_clean_final = (
    bls_joined
    .filter(pl.col("cbsa_code").is_not_null())
    .select([
        "year",
        "cbsa_code",
        "bls_msa_name",
        "area_title",
        "prim_state",
        "occ_code",
        "occ_title",
        "o_group",
        "tot_emp",
        "jobs_1000",
        "loc_quotient",
        "h_mean",
        "a_mean",
        "h_median",
        "a_median",
    ])
    .sort(["cbsa_code", "occ_code"])
)

unmatched_bls_metros = (
    bls_joined
    .filter(pl.col("cbsa_code").is_null())
    .select(["area_title", "metro_name_norm"])
    .unique()
    .sort("area_title")
)

unmatched_bls_metros.write_csv(UNMATCHED_BLS_METROS_CSV_PATH)
bls_clean_final.write_csv(BLS_CLEAN_CSV_PATH)

print_shape("BLS clean occupation × metro rows", bls_clean_final)
display_pl(bls_clean_final)

print(f"Unmatched BLS metros saved to: {UNMATCHED_BLS_METROS_CSV_PATH}")
print(f"Saved clean BLS file to:      {BLS_CLEAN_CSV_PATH}")

Sanitized raw BLS columns:
['area', 'area_title', 'area_type', 'prim_state', 'naics', 'naics_title', 'i_group', 'own_code', 'occ_code', 'occ_title', 'o_group', 'tot_emp', 'emp_prse', 'jobs_1000', 'loc_quotient', 'pct_total', 'pct_rpt', 'h_mean', 'a_mean', 'mean_prse', 'h_pct10', 'h_pct25', 'h_median', 'h_pct75', 'h_pct90', 'a_pct10', 'a_pct25', 'a_median', 'a_pct75', 'a_pct90', 'annual', 'hourly']

Unique sample values for area_type:
   area_type
0          6

Unique sample values for i_group:
          i_group
0  cross-industry

Unique sample values for own_code:
   own_code
0      1235

Unique sample values for naics:
   naics
0      0

Unique sample values for naics_title:
      naics_title
0  Cross-industry

Unique sample values for o_group:
    o_group
0  detailed
1     major
2     total
Kept cross-industry rows only: 45,677 rows
Code-based join coverage: 0.00%
Name-based join coverage: 0.00%
Using normalized metro-name join.
BLS clean occupation × metro rows: 0 rows × 15 cols


,year,cbsa_code,bls_msa_name,area_title,prim_state,occ_code,occ_title,o_group,tot_emp,jobs_1000,loc_quotient,h_mean,a_mean,h_median,a_median


Unmatched BLS metros saved to: /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/interim/bls_unmatched_metros.csv
Saved clean BLS file to:      /Users/omarelnesr/Desktop/Classes/CIS 2450/Final Project/data/interim/oews_2024_occ_metro.csv


## 10. Merge cleaned BLS and ACS tables

In [ ]:
merged = (
    bls_clean_final
    .join(
        acs_clean.select([
            "year",
            "cbsa_code",
            "median_gross_rent",
            "annual_rent",
            "median_household_income",
            "population",
        ]),
        on=["year", "cbsa_code"],
        how="left",
    )
    .with_columns([
        pl.when(pl.col("a_mean").is_not_null() & (pl.col("a_mean") > 0) & pl.col("annual_rent").is_not_null())
        .then(pl.col("annual_rent") / pl.col("a_mean"))
        .otherwise(None)
        .alias("rent_to_annual_mean_wage_ratio"),

        pl.when(pl.col("a_median").is_not_null() & (pl.col("a_median") > 0) & pl.col("annual_rent").is_not_null())
        .then(pl.col("annual_rent") / pl.col("a_median"))
        .otherwise(None)
        .alias("rent_to_annual_median_wage_ratio"),

        pl.when(pl.col("median_household_income").is_not_null() & (pl.col("median_household_income") > 0) & pl.col("a_mean").is_not_null())
        .then(pl.col("a_mean") / pl.col("median_household_income"))
        .otherwise(None)
        .alias("annual_mean_wage_to_metro_income_ratio"),

        pl.when(pl.col("tot_emp").is_not_null() & (pl.col("tot_emp") > 0))
        .then(pl.col("tot_emp").log())
        .otherwise(None)
        .alias("log_tot_emp"),
    ])
)

merged.write_csv(MERGED_CSV_PATH)

print_shape("Merged occupation × metro dataset", merged)
display_pl(merged)
print(f"Saved merged CSV: {MERGED_CSV_PATH}")

Merged occupation × metro dataset: 0 rows × 22 cols


,year,cbsa_code,bls_msa_name,area_title,occ_code,occ_title,occ_group,tot_emp,jobs_1000,loc_quotient,...,h_median,a_median,median_gross_rent,annual_rent,median_household_income,population,rent_to_annual_mean_wage_ratio,rent_to_annual_median_wage_ratio,annual_mean_wage_to_metro_income_ratio,log_tot_emp


TypeError: float() argument must be a string or a real number, not 'NoneType'

## 11. Quick diagnostic tables

In [ ]:

print("Top metros by number of occupation rows:")
display(
    merged.group_by("bls_msa_name")
    .agg(pl.len().alias("n_rows"))
    .sort("n_rows", descending=True)
    .head(15)
    .to_pandas()
)

print("Top occupations by total employment (where available):")
if "tot_emp" in merged.columns:
    display(
        merged.group_by("occ_title")
        .agg(pl.sum("tot_emp").alias("total_emp"))
        .sort("total_emp", descending=True)
        .head(15)
        .to_pandas()
    )
else:
    print("No total employment column was available in the BLS raw file.")


## 12. EDA setup

In [ ]:

eda = merged.filter(
    pl.col("annual_rent").is_not_null() &
    (
        (pl.col("a_mean").is_not_null()) |
        (pl.col("a_median").is_not_null())
    )
)

# Pick one wage column to use consistently in plots.
wage_col = "a_mean" if "a_mean" in eda.columns and eda.select(pl.col("a_mean").is_not_null().sum()).item() > 0 else "a_median"
ratio_col = "rent_to_annual_mean_wage_ratio" if wage_col == "a_mean" else "rent_to_annual_median_wage_ratio"

print(f"Using wage column for EDA: {wage_col}")
print(f"Using affordability ratio: {ratio_col}")

eda_pd = eda.select([
    "cbsa_code",
    "bls_msa_name",
    "occ_code",
    "occ_title",
    "annual_rent",
    wage_col,
    ratio_col,
    *[c for c in ["tot_emp", "jobs_1000", "loc_quotient"] if c in eda.columns],
]).to_pandas()

display(eda_pd.head())


## 13. Graph 1 — distribution of annual wages

In [ ]:

plot_df = eda_pd.dropna(subset=[wage_col]).copy()

plt.figure(figsize=(10, 6))
plt.hist(plot_df[wage_col], bins=40)
plt.title(f"Distribution of {wage_col} across occupation–metro rows (2024)")
plt.xlabel(wage_col)
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_wage_distribution.png", dpi=150)
plt.show()


## 14. Graph 2 — annual rent vs annual wage

In [ ]:

plot_df = eda_pd.dropna(subset=["annual_rent", wage_col]).copy()

# Sample if the dataset is very large, to keep plotting responsive.
if len(plot_df) > 20000:
    plot_df = plot_df.sample(20000, random_state=42)

plt.figure(figsize=(10, 6))
plt.scatter(plot_df["annual_rent"], plot_df[wage_col], alpha=0.25)
plt.title(f"Annual rent vs {wage_col} (2024 occupation–metro rows)")
plt.xlabel("Annual rent")
plt.ylabel(wage_col)
plt.tight_layout()
plt.savefig(FIG_DIR / "02_rent_vs_wage_scatter.png", dpi=150)
plt.show()


## 15. Pick a sample occupation for metro comparisons

In [ ]:

candidate_keywords = [
    "software developers",
    "registered nurses",
    "accountants",
    "data scientists",
    "financial analysts",
]

selected_occ = None
all_occ_titles = merged.select("occ_title").drop_nulls().unique().to_series().to_list()

for keyword in candidate_keywords:
    matches = [title for title in all_occ_titles if keyword in str(title).lower()]
    if matches:
        selected_occ = matches[0]
        break

if selected_occ is None:
    if "tot_emp" in merged.columns:
        selected_occ = (
            merged.group_by("occ_title")
            .agg(pl.sum("tot_emp").alias("total_emp"))
            .sort("total_emp", descending=True)
            .select("occ_title")
            .item()
        )
    else:
        selected_occ = merged.select("occ_title").drop_nulls().head(1).item()

print("Selected occupation for metro-level comparison:")
print(selected_occ)


## 16. Graph 3 — top metros for a selected occupation by affordability

In [ ]:

occ_df = (
    merged
    .filter(pl.col("occ_title") == selected_occ)
    .filter(pl.col(ratio_col).is_not_null())
    .sort(ratio_col)
    .head(15)
    .select(["bls_msa_name", wage_col, "annual_rent", ratio_col, *[c for c in ["tot_emp", "loc_quotient"] if c in merged.columns]])
    .to_pandas()
)

display(occ_df)

plt.figure(figsize=(10, 7))
plt.barh(occ_df["bls_msa_name"], occ_df[ratio_col])
plt.title(f"Top metros for {selected_occ} by rent-to-wage ratio (lower is better)")
plt.xlabel(ratio_col)
plt.ylabel("Metro area")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIG_DIR / "03_selected_occupation_affordability.png", dpi=150)
plt.show()


## 17. Graph 4 — location quotient vs annual wage for the selected occupation

In [ ]:

if "loc_quotient" in merged.columns:
    lq_df = (
        merged
        .filter(pl.col("occ_title") == selected_occ)
        .filter(pl.col("loc_quotient").is_not_null() & pl.col(wage_col).is_not_null())
        .select(["bls_msa_name", "loc_quotient", wage_col, *[c for c in ["tot_emp"] if c in merged.columns]])
        .to_pandas()
    )

    display(lq_df.head())

    plt.figure(figsize=(10, 6))
    plt.scatter(lq_df["loc_quotient"], lq_df[wage_col], alpha=0.5)
    plt.title(f"Location quotient vs {wage_col} for {selected_occ}")
    plt.xlabel("Location quotient")
    plt.ylabel(wage_col)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "04_selected_occupation_lq_vs_wage.png", dpi=150)
    plt.show()
else:
    print("The BLS raw file did not expose a location quotient column after cleaning.")


## 18. Graph 5 — distribution of affordability ratios for high-employment occupations

In [ ]:

if "tot_emp" in merged.columns:
    top_occ = (
        merged.group_by("occ_title")
        .agg(pl.sum("tot_emp").alias("total_emp"))
        .sort("total_emp", descending=True)
        .head(6)
        .select("occ_title")
        .to_series()
        .to_list()
    )

    box_df = (
        merged
        .filter(pl.col("occ_title").is_in(top_occ))
        .filter(pl.col(ratio_col).is_not_null())
        .select(["occ_title", ratio_col])
        .to_pandas()
    )

    grouped_values = [box_df.loc[box_df["occ_title"] == occ, ratio_col].values for occ in top_occ]

    plt.figure(figsize=(12, 6))
    plt.boxplot(grouped_values, tick_labels=top_occ, vert=True)
    plt.title("Affordability ratio distribution for high-employment occupations")
    plt.ylabel(ratio_col)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "05_affordability_boxplot_top_occupations.png", dpi=150)
    plt.show()
else:
    print("The BLS raw file did not expose a total employment column after cleaning.")


## 19. Final output summary

In [ ]:

summary = {
    "raw_bls_csv": str(BLS_RAW_CSV_PATH),
    "raw_acs_csv": str(ACS_RAW_CSV_PATH),
    "clean_bls_csv": str(BLS_CLEAN_CSV_PATH),
    "clean_acs_csv": str(ACS_CLEAN_CSV_PATH),
    "merged_csv": str(MERGED_CSV_PATH),
    "metro_crosswalk_csv": str(BLS_CROSSWALK_CSV_PATH),
    "figures_dir": str(FIG_DIR),
    "unmatched_bls_metros_csv": str(UNMATCHED_BLS_METROS_CSV_PATH),
}

display(pd.DataFrame(summary.items(), columns=["artifact", "path"]))

print("\nIf the merge coverage is weaker than expected, inspect these first:")
print(f"  - raw BLS columns in section 9")
print(f"  - {UNMATCHED_BLS_METROS_CSV_PATH}")
print("  - the code-join vs name-join coverage printed in section 9")
